# TP 2 · Titanic — Identification et création des bonnes variables

**Jour 2 — chapitre 03 : Feature engineering**

## Mise en situation

Sur la base des features nettoyées ce matin (notebook précédent), il s'agit maintenant de
sélectionner et d'enrichir les variables avant la modélisation.


In [7]:
import pandas as pd
import seaborn as sns

titanic_raw = sns.load_dataset("titanic")
titanic = titanic_raw.rename(columns={
    "survived": "Survived",
    "pclass": "Pclass",
    "sex": "Sex",
    "age": "Age",
    "sibsp": "SibSp",
    "parch": "Parch",
    "fare": "Fare",
    "embarked": "Embarked",
    "who": "Who",
})
titanic = titanic[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()
titanic.head()

# On reprend le nettoyage du TP precedent (imputation Age / Embarked)
age_median_par_groupe = titanic.groupby(["Sex", "Pclass"])["Age"].transform("median")
titanic["Age"] = titanic["Age"].fillna(age_median_par_groupe).fillna(titanic["Age"].median())
titanic["Embarked"] = titanic["Embarked"].fillna(titanic["Embarked"].mode()[0])
titanic.isnull().sum()


Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

## Étape 1 — Encoder les variables catégorielles (Sex, Embarked)

**Question de réflexion :** `Sex` et `Embarked` sont des catégories sans ordre naturel.
Quelle technique d'encodage utiliser, et pourquoi pas un simple encodage numérique
(0, 1, 2) ?


In [8]:
titanic_encoded = pd.get_dummies(titanic, columns=["Sex", "Embarked"], drop_first=True, dtype=int)
titanic_encoded.head()


# Ou manualement par 
# titanic["Sex"] = titanic["Sex"].map({"female": 0, "male": 1})
#titanic["Embarked"] = titanic["Embarked"].map({"C": 0, "Q": 1, "S": 2})


,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,1,0,1
1,1,1,38.0,1,0,71.2833,0,0,0
2,1,3,26.0,0,0,7.9250,0,0,1
3,1,1,35.0,1,0,53.1000,0,0,1
4,0,3,35.0,0,0,8.0500,1,0,1


**Ce qu'on observe** : le one-hot encoding crée une colonne binaire par catégorie (moins
une, avec `drop_first=True`, pour éviter la redondance). Un simple label encoding (0, 1, 2)
aurait introduit un ordre artificiel entre des catégories qui n'en ont pas.


## Étape 2 — Créer la variable FamilySize et la variable IsAlone

**Question de réflexion :** comment combiner `SibSp` (frères/soeurs/conjoints à bord) et
`Parch` (parents/enfants à bord) en une seule variable pertinente ? Et comment en déduire
si le passager voyageait seul ?


In [9]:
titanic_encoded["FamilySize"] = titanic_encoded["SibSp"] + titanic_encoded["Parch"] + 1
titanic_encoded["IsAlone"] = (titanic_encoded["FamilySize"] == 1).astype(int)

titanic_encoded[["SibSp", "Parch", "FamilySize", "IsAlone"]].head()


,SibSp,Parch,FamilySize,IsAlone
0,1,0,2,0
1,1,0,2,0
2,0,0,1,1
3,1,0,2,0
4,0,0,1,1


**Ce qu'on observe** : `FamilySize` compte le passager lui-même (+1), plus ses proches à
bord. `IsAlone` vaut 1 quand `FamilySize` vaut exactement 1. Ces deux variables sont des
exemples typiques de features « combinées », créées à partir de variables existantes.


## Étape 3 — Extraire le titre social depuis le nom

**Question de réflexion :** le dataset seaborn ne fournit pas la colonne `Name`. Si vous
aviez le vrai `train.csv` Kaggle, comment extrairiez-vous un titre (Mr, Mrs, Miss, Master)
depuis une chaîne de caractères comme `"Braund, Mr. Owen Harris"` ? Et en quoi ce titre
est-il une feature potentiellement plus informative que l'âge brut lui-même ?


In [10]:
# Demonstration sur des exemples de noms, au format Kaggle ("Nom de famille, Titre. Prenom")
exemples_noms = pd.Series([
    "Braund, Mr. Owen Harris",
    "Cumings, Mrs. John Bradley (Florence Briggs Thayer)",
    "Heikkinen, Miss. Laina",
    "Palsson, Master. Gosta Leonard",
])

titres_extraits = exemples_noms.str.extract(r" ([A-Za-z]+)\.")
pd.DataFrame({"Name": exemples_noms, "Title": titres_extraits[0]})


,Name,Title
0,"Braund, Mr. Owen Harris",Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs
2,"Heikkinen, Miss. Laina",Miss
3,"Palsson, Master. Gosta Leonard",Master


**Ce qu'on observe** : `str.extract` avec une expression régulière isole le mot situé
juste avant le point, qui correspond au titre social. Ce titre est un excellent proxy :
il encode à la fois l'âge approximatif (Master désigne un jeune garçon), le sexe et parfois
le statut social (Mrs vs Miss), en une seule variable catégorielle facile à encoder.

Sur le vrai dataset Kaggle : `titanic["Title"] = titanic["Name"].str.extract(r" ([A-Za-z]+)\.")`,
suivi d'un regroupement des titres rares (Dr, Rev, Col...) dans une catégorie `"Rare"`.


## Étape 4 — Sélectionner un premier jeu de features candidates

**Question de réflexion :** parmi toutes les variables désormais disponibles, lesquelles
retenir pour le prochain modèle ? Faut-il déjà tout inclure ?


In [11]:
features_candidates = [
    "Pclass", "Age", "Fare", "FamilySize", "IsAlone",
    "Sex_male",
]
features_candidates = [c for c in features_candidates if c in titanic_encoded.columns]

X = titanic_encoded[features_candidates]
y = titanic_encoded["Survived"]

X.head()


,Pclass,Age,Fare,FamilySize,IsAlone,Sex_male
0,3,22.0,7.2500,2,0,1
1,1,38.0,71.2833,2,0,0
2,3,26.0,7.9250,1,1,0
3,1,35.0,53.1000,2,0,0
4,3,35.0,8.0500,1,1,1


**Ce qu'on observe** : on part d'un jeu de features raisonnable, ni trop pauvre (on garde
les variables identifiées comme discriminantes en jour 1 : sexe, classe), ni pléthorique.
La sélection plus rigoureuse (filtre, wrapper, embarqué, vue en cours) pourra affiner ce
choix ensuite. Les deux étapes suivantes entraînent un modèle sur ces features et vérifient s'il
fait mieux que le premier modèle du notebook 01, construit sans `Age` ni variables créées.


## Étape 5 — Entraîner un modèle sur ces features

**Question de réflexion :** comment savoir si les variables créées servent vraiment à quelque
chose ? Sur quel découpage train / test faut-il entraîner ce modèle pour que la comparaison
avec le premier modèle du notebook 01 reste honnête ?

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# random_state=42 : exactement le meme decoupage que le premier modele du notebook 01
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modele = LogisticRegression(max_iter=5000)
modele.fit(X_train, y_train)

predictions = modele.predict(X_test)

# Dix premiers passagers du jeu de test : prediction du modele face a la realite
apercu = X_test.head(10).copy()
apercu["Survived_reel"] = y_test.head(10)
apercu["Survived_predit"] = predictions[:10]
apercu[["Survived_reel", "Survived_predit"]]

,Survived_reel,Survived_predit
709,1,0
439,0,0
840,0,0
720,1,1
39,1,1
290,1,1
300,1,1
333,0,0
208,1,1
136,1,1


**Ce qu'on observe** : le modèle s'entraîne sur 80 % des passagers et prédit sur les 20 %
qu'il n'a jamais vus. Rien de nouveau par rapport au notebook 01 côté code : seule la matrice
`X` a changé, elle contient maintenant `Age` (imputé), `FamilySize` et `IsAlone`. C'est
précisément ce qu'on veut isoler : même modèle, même découpage, autres variables.

## Étape 6 — Évaluer : le feature engineering a-t-il payé ?

**Question de réflexion :** à quoi comparer ce modèle pour répondre ? Et pourquoi ne pas se
contenter de l'accuracy sur un jeu où seuls 38 % des passagers survivent ?

In [13]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Modele de reference : les 5 colonnes brutes du notebook 01, memes passagers, meme decoupage
features_notebook_01 = ["Pclass", "Sex_male", "SibSp", "Parch", "Fare"]
X_reference = titanic_encoded[features_notebook_01]

modele_reference = LogisticRegression(max_iter=5000)
modele_reference.fit(X_reference.loc[X_train.index], y_train)
predictions_reference = modele_reference.predict(X_reference.loc[X_test.index])

def metriques(y_reel, y_predit):
    return {
        "accuracy": accuracy_score(y_reel, y_predit),
        "precision": precision_score(y_reel, y_predit),
        "recall": recall_score(y_reel, y_predit),
        "f1": f1_score(y_reel, y_predit),
    }

pd.DataFrame({
    "Sans feature engineering (notebook 01)": metriques(y_test, predictions_reference),
    "Avec feature engineering": metriques(y_test, predictions),
}).round(3)

,Sans feature engineering (notebook 01),Avec feature engineering
accuracy,0.788,0.810
precision,0.765,0.803
recall,0.703,0.716
f1,0.732,0.757


**Ce qu'on observe** : les quatre métriques progressent, d'environ 2 points d'accuracy et
2,5 points de F1. Le gain est réel mais modeste : le feature engineering affine un modèle,
il ne le transforme pas. Et il faut regarder au-delà de l'accuracy — sur un jeu déséquilibré,
c'est le F1 (équilibre entre précision et rappel sur les survivants) qui dit si le modèle
progresse vraiment sur la classe qui nous intéresse.

Attention enfin à la limite de l'exercice : un seul découpage train / test, donc un seul
chiffre par métrique. Le notebook 01 a montré à quel point ce chiffre bouge d'un `random_state`
à l'autre. Pour trancher proprement, c'est la cross-validation qu'il faudrait utiliser.